In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso
import warnings

warnings.filterwarnings('ignore')

from multiprocessing import get_context
import tensorflow as tf
import time
from decimal import Decimal, ROUND_HALF_UP
from scipy.optimize import brentq

import scipy.linalg as sl


def load_yearly_signals(year, buys_path_template='buys_{}.csv', sells_path_template='sells_{}.csv'):
    """
    Load buy and sell signals for a specific year.
    
    Parameters:
    -----------
    year : int
        Year to load signals for
    buys_path_template : str
        Template for buys file path (use {} for year placeholder)
    sells_path_template : str
        Template for sells file path (use {} for year placeholder)
    
    Returns:
    --------
    permno_set : set
        Set of permnos in the buy and sell signals for this year
    """
    try:
        buys = pd.read_csv(buys_path_template.format(year), index_col=1)
        sells = pd.read_csv(sells_path_template.format(year), index_col=1)
        
        buys.index.name = 'permno'
        sells.index.name = 'permno'
        
        buys_index = buys.index.astype(int)
        sells_index = sells.index.astype(int)
        
        return set(buys_index.union(sells_index))
    except FileNotFoundError as e:
        print(f"  ⚠ Warning: Could not load signals for year {year}: {e}")
        return set()

def load_finbert_signals(signals_path):
    """Load FinBERT monthly signals from CSV file."""
    try:
        signals_df = pd.read_csv(signals_path)
        signals_df['date'] = pd.to_datetime(signals_df['year_month']) + pd.offsets.MonthEnd(0)
        return signals_df
    except FileNotFoundError as e:
        print(f"  ⚠ Warning: Could not load FinBERT signals: {e}")
        return pd.DataFrame(columns=['symbol', 'company', 'year_month', 'signal', 'date'])


def get_finbert_permnos_for_date(signals_df, ticker_to_permno, date):
    """Get set of permnos with 'buy' or 'sell' signals for a specific date."""
    date_signals = signals_df[signals_df['date'] == date]
    buy_signals = date_signals[date_signals['signal'] == 'buy']
    sell_signals = date_signals[date_signals['signal'] == 'sell']
    
    permnos = set()
    for ticker in buy_signals['symbol'].values:
        if ticker in ticker_to_permno:
            permnos.add(ticker_to_permno[ticker])
    for ticker in sell_signals['symbol'].values:
        if ticker in ticker_to_permno:
            permnos.add(ticker_to_permno[ticker])
    
    return permnos


def create_ticker_to_permno_mapping(df):
    """Create a mapping from ticker to permno."""
    if 'ticker' not in df.columns:
        raise ValueError("DataFrame must have 'ticker' column for mapping")
    
    valid_df = df[df['ticker'].notna()].copy()
    ticker_to_permno = valid_df.groupby('ticker')['permno'].last().to_dict()
    
    return ticker_to_permno


def calculate_exit_transaction_cost(prev_weights_dict, prev_oos_returns_dict, 
                                    prev_gross_return, transaction_cost, verbose=False):
    """Calculate transaction cost when exiting the market (liquidating all positions)."""
    if len(prev_weights_dict) == 0:
        return 0.0, 0.0, 0.0
    
    adjusted_prev = {}
    for asset, prev_w in prev_weights_dict.items():
        if asset in prev_oos_returns_dict:
            prev_r = prev_oos_returns_dict[asset]
            if abs(1 + prev_gross_return) > 1e-6:
                adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_return)
            else:
                adjusted_prev[asset] = 0.0
        else:
            if abs(1 + prev_gross_return) > 1e-6:
                adjusted_prev[asset] = prev_w / (1 + prev_gross_return)
            else:
                adjusted_prev[asset] = 0.0
    
    turnover = sum(abs(w) for w in adjusted_prev.values())
    tc = transaction_cost * 1.0 * turnover
    net_return = -tc
    
    if verbose:
        print(f"  Liquidating positions | Turnover: {turnover:>6.4f} | TC: {tc:>8.6f}")
    
    return turnover, tc, net_return


2026-05-29 15:37:38.158316: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def backtest_equal_weight_yearly(df, 
                                 test_start_date='2020-01-31', 
                                 test_end_date='2024-11-30',
                                 lookback_window=180,
                                 transaction_cost=0.001,
                                 buys_path_template='buys_{}.csv',
                                 sells_path_template='sells_{}.csv',
                                 finbert_signals_path=None,
                                 data_factor=None,
                                 verbose=True):
    """
    Backtest the yearly + FinBERT signal set as an equal-weight portfolio.

    The signal filtering logic stays the same, but no POET / MV / MSR optimization
    is applied. Each period uses equal weights across the investable assets.
    """
    df = df.copy()
    if 'datadate' not in df.columns or 'permno' not in df.columns:
        raise ValueError("DataFrame must have 'datadate' and 'permno' columns")
    df['datadate'] = pd.to_datetime(df['datadate'])

    if data_factor is not None:
        _ = data_factor.shape

    if verbose:
        print("Creating ticker to permno mapping...")
    ticker_to_permno = create_ticker_to_permno_mapping(df)
    if verbose:
        print(f"Mapped {len(ticker_to_permno)} unique tickers to permnos")

    finbert_df = None
    if finbert_signals_path is not None:
        finbert_df = load_finbert_signals(finbert_signals_path)
        if verbose and len(finbert_df) > 0:
            print(f"Loaded FinBERT signals: {len(finbert_df)} monthly records")
            print("FinBERT signal distribution:")
            print(finbert_df['signal'].value_counts())

    all_dates = sorted(df['datadate'].unique())
    test_start_dt = pd.to_datetime(test_start_date)
    test_end_dt = pd.to_datetime(test_end_date)

    try:
        test_start_idx = all_dates.index(test_start_dt)
        test_end_idx = all_dates.index(test_end_dt)
    except ValueError as e:
        raise ValueError(f"Date not found in DataFrame: {e}")

    portfolio_returns = []
    portfolio_dates = []
    portfolio_weights_list = []
    portfolio_turnover_list = []
    portfolio_gross_returns = []

    prev_weights_dict = {}
    prev_oos_returns_dict = {}
    prev_gross_return = 0.0

    yearly_signals_cache = {}

    def calculate_rebalance_transaction_cost(new_weights_dict, gross_return):
        if len(prev_weights_dict) == 0:
            turnover = sum(abs(w) for w in new_weights_dict.values())
            tc = transaction_cost * (1 + gross_return) * turnover
            return turnover, tc

        adjusted_prev = {}
        for asset, prev_w in prev_weights_dict.items():
            if asset in prev_oos_returns_dict and abs(1 + prev_gross_return) > 1e-6:
                prev_r = prev_oos_returns_dict[asset]
                adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_return)
            elif abs(1 + prev_gross_return) > 1e-6:
                adjusted_prev[asset] = prev_w / (1 + prev_gross_return)
            else:
                adjusted_prev[asset] = 0.0

        all_assets = set(adjusted_prev.keys()) | set(new_weights_dict.keys())
        turnover = 0.0
        for asset in all_assets:
            turnover += abs(new_weights_dict.get(asset, 0.0) - adjusted_prev.get(asset, 0.0))
        tc = transaction_cost * (1 + gross_return) * turnover
        return turnover, tc

    if verbose:
        print("=" * 60)
        print("STARTING BACKTEST WITH EQUAL-WEIGHT YEARLY + FINBERT SIGNALS")
        print("=" * 60)

    for t in range(test_start_idx, test_end_idx + 1):
        current_date = all_dates[t]
        current_year = current_date.year

        if current_year not in yearly_signals_cache:
            yearly_signals_cache[current_year] = load_yearly_signals(
                current_year, buys_path_template, sells_path_template
            )

        yearly_permnos = yearly_signals_cache[current_year]

        finbert_permnos = set()
        if finbert_df is not None and len(finbert_df) > 0:
            finbert_permnos = get_finbert_permnos_for_date(finbert_df, ticker_to_permno, current_date)

        allowed_permnos = yearly_permnos.intersection(finbert_permnos)
        if len(allowed_permnos) <= 1:
            allowed_permnos = yearly_permnos.union(finbert_permnos)

        oos_data = df[(df['datadate'] == current_date) & (df['permno'].isin(allowed_permnos))]
        oos_returns_series = oos_data.set_index('permno')['ret_fwd_1'].dropna()
        oos_returns_dict = oos_returns_series.to_dict()

        if len(allowed_permnos) == 0:
            if verbose:
                print(f"\n[{t - test_start_idx + 1}/{test_end_idx - test_start_idx + 1}] "
                      f"Date: {current_date.strftime('%Y-%m-%d')}")
                print("  ⚠ No signals, recording zero return")

            turnover, tc, net_return = calculate_exit_transaction_cost(
                prev_weights_dict, prev_oos_returns_dict, prev_gross_return, transaction_cost,
                verbose=verbose
            )

            portfolio_returns.append(net_return)
            portfolio_dates.append(current_date)
            portfolio_weights_list.append({})
            portfolio_turnover_list.append(turnover)
            portfolio_gross_returns.append(0.0)

            prev_weights_dict = {}
            prev_oos_returns_dict = {}
            prev_gross_return = 0.0
            continue

        window_start_date = all_dates[t - lookback_window]
        window_end_date = all_dates[t - 1]

        train_data = df[(df['datadate'] >= window_start_date) &
                        (df['datadate'] <= window_end_date) &
                        (df['permno'].isin(allowed_permnos))]

        returns_pivot = train_data.pivot(index='datadate', columns='permno', values='ret_fwd_1')
        window_dates = all_dates[t - lookback_window : t]
        returns_pivot = returns_pivot.reindex(index=window_dates)

        filtered_pivot = returns_pivot.drop(columns=returns_pivot.columns[returns_pivot.isna().any()])
        current_assets = [asset for asset in filtered_pivot.columns if asset in oos_returns_dict]
        Y = filtered_pivot[current_assets].values
        n_train, p_current = Y.shape

        if verbose:
            print(f"\n[{t - test_start_idx + 1}/{test_end_idx - test_start_idx + 1}] "
                  f"Date: {current_date.strftime('%Y-%m-%d')} | Year: {current_year}")
            print(f"  Window: {window_start_date.strftime('%Y-%m-%d')} to {window_end_date.strftime('%Y-%m-%d')}")
            print(f"  Yearly: {len(yearly_permnos)} | FinBERT: {len(finbert_permnos)} | "
                  f"Allowed: {len(allowed_permnos)} | Assets w/ data: {p_current}")

        if n_train < lookback_window or p_current < 1:
            if verbose:
                print(f"  ⚠ Insufficient data (n={n_train}, p={p_current}), recording 0 return")

            turnover, tc, net_return = calculate_exit_transaction_cost(
                prev_weights_dict, prev_oos_returns_dict, prev_gross_return, transaction_cost,
                verbose=verbose
            )

            portfolio_returns.append(net_return)
            portfolio_dates.append(current_date)
            portfolio_weights_list.append({})
            portfolio_turnover_list.append(turnover)
            portfolio_gross_returns.append(0.0)

            prev_weights_dict = {}
            prev_oos_returns_dict = {}
            prev_gross_return = 0.0
            continue

        equal_weight = 1.0 / p_current
        new_weights_dict = {asset: equal_weight for asset in current_assets}
        gross_return = sum(new_weights_dict[a] * oos_returns_dict[a] for a in current_assets)
        turnover, tc = calculate_rebalance_transaction_cost(new_weights_dict, gross_return)
        net_return = gross_return - tc

        portfolio_returns.append(net_return)
        portfolio_dates.append(current_date)
        portfolio_weights_list.append(new_weights_dict.copy())
        portfolio_turnover_list.append(turnover)
        portfolio_gross_returns.append(gross_return)

        prev_weights_dict = new_weights_dict.copy()
        prev_oos_returns_dict = {asset: oos_returns_dict[asset] for asset in current_assets}
        prev_gross_return = gross_return

        if verbose:
            print(f"  Equal-weight gross: {gross_return:>8.5f} | Turnover: {turnover:>6.4f} | "
                  f"TC: {tc:>8.6f} | Net: {net_return:>8.5f}")

    if verbose:
        print("\n" + "=" * 60)
        print("BACKTEST COMPLETE")
        print("=" * 60)

    results_df = pd.DataFrame({
        'date': portfolio_dates,
        'portfolio_return': portfolio_returns,
        'portfolio_gross_return': portfolio_gross_returns,
        'portfolio_weights': portfolio_weights_list,
        'portfolio_turnover': portfolio_turnover_list
    })
    results_df['cumulative_return'] = (1 + results_df['portfolio_return']).cumprod() - 1

    def compute_metrics(returns_list, turnover_list, results_df):
        if len(returns_list) > 0:
            mean_return = np.mean(returns_list)
            variance = np.var(returns_list, ddof=1)
            sharpe_ratio = mean_return / np.sqrt(variance) if variance > 0 else 0
            annual_return = mean_return * 12
            annual_volatility = np.sqrt(variance * 12)
            annual_sharpe = annual_return / annual_volatility if annual_volatility > 0 else 0
            return {
                'mean_return': mean_return,
                'variance': variance,
                'sharpe_ratio': sharpe_ratio,
                'annual_return': annual_return,
                'annual_volatility': annual_volatility,
                'annual_sharpe_ratio': annual_sharpe,
                'total_return': results_df['cumulative_return'].iloc[-1],
                'avg_turnover': np.mean(turnover_list),
                'n_periods': len(returns_list)
            }
        return {
            'mean_return': 0, 'variance': 0, 'sharpe_ratio': 0,
            'annual_return': 0, 'annual_volatility': 0, 'annual_sharpe_ratio': 0,
            'total_return': 0, 'avg_turnover': 0, 'n_periods': 0
        }

    metrics = compute_metrics(portfolio_returns, portfolio_turnover_list, results_df)
    results_df_2 = results_df.copy()
    results_df_3 = results_df.copy()
    metrics_2 = metrics.copy()
    metrics_3 = metrics.copy()

    return results_df, metrics, results_df_2, metrics_2, results_df_3, metrics_3

In [4]:
df = pd.read_csv('../../green cleaned.csv', dtype={'ncusip': 'string'})
df['ret_fwd_1'] = df.groupby('permno')['ret_excess'].shift(-1)

data_f = pd.read_csv('F-F_Research_Data_Factors.csv', sep=',')
data_f['Date'] = pd.to_datetime(data_f['Date'], format="%Y%m")
data_f['Date'] = data_f['Date'] + pd.offsets.MonthEnd(0)
data_f = data_f.set_index('Date')
data_f = data_f[['Mkt-RF', 'SMB', 'HML', 'RF']].astype(float)

# Run equal-weight backtest with yearly + FinBERT signals
results_df, metrics, results_df_2, metrics_2, results_df_3, metrics_3 = backtest_equal_weight_yearly(
    df,
    test_start_date='2015-01-31',
    test_end_date='2024-04-30',
    lookback_window=180,
    transaction_cost=0.001,
    buys_path_template='buys_{}.csv',
    sells_path_template='sells_{}.csv',
    finbert_signals_path='../examples/monthly_signals_decay.csv',
    data_factor=data_f,
    verbose=True
)

print("\nEqual-weighted allowed_permnos portfolio")
print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.4f}")
print(f"Annualized Sharpe Ratio: {metrics['annual_sharpe_ratio']:.4f}")
print(f"Total Return: {metrics['total_return']:.4f}")
print(f"Average Turnover: {metrics['avg_turnover']:.4f}")

Creating ticker to permno mapping...
Mapped 1664 unique tickers to permnos
Loaded FinBERT signals: 54240 monthly records
FinBERT signal distribution:
signal
hold    52823
sell      859
buy       558
Name: count, dtype: int64
STARTING BACKTEST WITH EQUAL-WEIGHT YEARLY + FINBERT SIGNALS

[1/112] Date: 2015-01-31 | Year: 2015
  Window: 2000-01-31 to 2014-12-31
  Yearly: 54 | FinBERT: 10 | Allowed: 63 | Assets w/ data: 35
  Equal-weight gross:  0.01986 | Turnover: 1.0000 | TC: 0.001020 | Net:  0.01884

[2/112] Date: 2015-02-28 | Year: 2015
  Window: 2000-02-29 to 2015-01-31
  Yearly: 54 | FinBERT: 4 | Allowed: 58 | Assets w/ data: 32
  Equal-weight gross: -0.02799 | Turnover: 0.3030 | TC: 0.000295 | Net: -0.02828

[3/112] Date: 2015-03-31 | Year: 2015
  Window: 2000-03-31 to 2015-02-28
  Yearly: 54 | FinBERT: 4 | Allowed: 58 | Assets w/ data: 32
  Equal-weight gross:  0.01427 | Turnover: 0.1571 | TC: 0.000159 | Net:  0.01411

[4/112] Date: 2015-04-30 | Year: 2015
  Window: 2000-04-30 to 20

In [5]:
print("\nEqual-weighted allowed_permnos portfolio")
print(f"Annualized Sharpe Ratio: {metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {metrics['mean_return']*12:.4f}")
print(f"Variance: {metrics['variance']*12:.4f}")
print(f"Avg Turnover: {metrics['avg_turnover']:.4f}")


Equal-weighted allowed_permnos portfolio
Annualized Sharpe Ratio: 0.4251
Mean Return: 0.1170
Variance: 0.0758
Avg Turnover: 0.9978


In [6]:
results_df['portfolio_return'].to_csv('../../LLM-S FinBERT no quant 10 years.csv', index=False)